In [ ]:
!pip install -q sentence-transformers faiss-cpu datasets google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 64.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np

from datasets import load_dataset

import os

from sentence_transformers import SentenceTransformer

import faiss

import google.generativeai as genai

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
dataset = load_dataset(
    "CShorten/ML-ArXiv-Papers",
    split="train"
)

print(dataset)

README.md:   0%|          | 0.00/986 [00:00<?, ?B/s]

ML-Arxiv-Papers.csv:   0%|          | 0.00/147M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/117592 [00:00<?, ? examples/s]

Dataset({
    features: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'],
    num_rows: 117592
})


In [ ]:
print(dataset[0])

{'Unnamed: 0.1': 0, 'Unnamed: 0': 0.0, 'title': 'Learning from compressed observations', 'abstract': '  The problem of statistical learning is to construct a predictor of a random\nvariable $Y$ as a function of a related random variable $X$ on the basis of an\ni.i.d. training sample from the joint distribution of $(X,Y)$. Allowable\npredictors are drawn from some specified class, and the goal is to approach\nasymptotically the performance (expected loss) of the best predictor in the\nclass. We consider the setting in which one has perfect observation of the\n$X$-part of the sample, while the $Y$-part has to be communicated at some\nfinite bit rate. The encoding of the $Y$-values is allowed to depend on the\n$X$-values. Under suitable regularity conditions on the admissible predictors,\nthe underlying family of probability distributions and the loss function, we\ngive an information-theoretic characterization of achievable predictor\nperformance in terms of conditional distortion-rate f

In [ ]:
df = pd.DataFrame(dataset)

df.head()

,Unnamed: 0.1,Unnamed: 0,title,abstract
0,0,0.0,Learning from compressed observations,The problem of statistical learning is to co...
1,1,1.0,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,2,2.0,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,3,3.0,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,4,4.0,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...


In [ ]:
print(df.shape)

(117592, 4)


In [ ]:
df.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'], dtype='object')

In [ ]:
df[['title', 'abstract']].head()

,title,abstract
0,Learning from compressed observations,The problem of statistical learning is to co...
1,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...


In [ ]:
df[['title','abstract']].isnull().sum()

,0
title,0
abstract,0


In [ ]:
print("Duplicate Papers:", df.duplicated().sum())

Duplicate Papers: 0


In [ ]:
df = df.drop_duplicates()

print("New Shape:", df.shape)

New Shape: (117592, 4)


In [ ]:
df["paper_text"] = df["title"] + ". " + df["abstract"]

In [ ]:
df[["title", "paper_text"]].head()

,title,paper_text
0,Learning from compressed observations,Learning from compressed observations. The p...
1,Sensor Networks with Random Links: Topology De...,Sensor Networks with Random Links: Topology De...
2,The on-line shortest path problem under partia...,The on-line shortest path problem under partia...
3,A neural network approach to ordinal regression,A neural network approach to ordinal regressio...
4,Parametric Learning and Monte Carlo Optimization,Parametric Learning and Monte Carlo Optimizati...


In [ ]:
print(df["paper_text"].iloc[0])

Learning from compressed observations.   The problem of statistical learning is to construct a predictor of a random
variable $Y$ as a function of a related random variable $X$ on the basis of an
i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable
predictors are drawn from some specified class, and the goal is to approach
asymptotically the performance (expected loss) of the best predictor in the
class. We consider the setting in which one has perfect observation of the
$X$-part of the sample, while the $Y$-part has to be communicated at some
finite bit rate. The encoding of the $Y$-values is allowed to depend on the
$X$-values. Under suitable regularity conditions on the admissible predictors,
the underlying family of probability distributions and the loss function, we
give an information-theoretic characterization of achievable predictor
performance in terms of conditional distortion-rate functions. The ideas are
illustrated on the example of nonparametric regres

In [ ]:
df["paper_length"] = df["paper_text"].apply(len)

df["paper_length"].describe()

,paper_length
count,117592.000000
mean,1232.876777
std,347.607683
min,70.000000
25%,989.000000
50%,1217.000000
75%,1466.000000
max,3394.000000


In [ ]:
df.to_csv("cleaned_research_papers.csv", index=False)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [ ]:
from sentence_transformers import SentenceTransformer

print("Loading Sentence Transformer Model...")

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model Loaded Successfully!")

Loading Sentence Transformer Model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model Loaded Successfully!


In [ ]:
print("Embedding Dimension:", model.get_sentence_embedding_dimension())

Embedding Dimension: 384


/tmp/ipykernel_13019/2756718674.py:1: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding Dimension:", model.get_sentence_embedding_dimension())


In [ ]:
sample_embedding = model.encode(df["paper_text"].iloc[0])

print(sample_embedding)

[-1.30282551e-01 -7.23162899e-03 -4.06495156e-03  3.25140990e-02
  1.11174710e-01  1.13913156e-02  1.01188108e-01 -8.72508734e-02
  4.50674184e-02 -1.33295646e-02 -3.44884917e-02  6.94484338e-02
  1.04651175e-01 -2.62786895e-02 -2.47473586e-02 -4.81175724e-03
  7.20172971e-02  5.97533695e-02 -1.14404686e-01  1.75182261e-02
  5.53538464e-02  2.09925305e-02  3.03000249e-02  3.27093564e-02
  1.02124281e-01 -9.46467593e-02  2.98457630e-02 -2.17609201e-02
 -4.44467627e-02 -1.01951975e-02 -2.31588446e-02 -4.75827381e-02
  9.64802131e-03 -3.65692824e-02  2.13589724e-02  3.16553302e-02
  2.98589487e-02  4.31113653e-02  4.72870991e-02  4.07869779e-02
 -8.61465815e-05  8.16625077e-03 -1.28682302e-02  3.26071382e-02
 -1.18403127e-02  3.88188586e-02  5.99052906e-02 -2.48664487e-02
 -7.29222968e-02  5.72386943e-02 -1.85121633e-02  7.00094402e-02
 -6.14270344e-02  6.39118953e-03  4.15449438e-04 -3.60912159e-02
 -2.26465389e-02 -1.75704509e-02 -4.75126468e-02  1.43809706e-01
 -3.72586846e-02 -2.48913

In [ ]:
print(sample_embedding.shape)

(384,)


In [ ]:
print("First 20 values:\n")

print(sample_embedding[:20])

First 20 values:

[-0.13028255 -0.00723163 -0.00406495  0.0325141   0.11117471  0.01139132
  0.10118811 -0.08725087  0.04506742 -0.01332956 -0.03448849  0.06944843
  0.10465118 -0.02627869 -0.02474736 -0.00481176  0.0720173   0.05975337
 -0.11440469  0.01751823]


In [ ]:


df = df.head(10000).copy()

print("Number of papers:", len(df))

Number of papers: 10000


In [ ]:
import numpy as np

In [ ]:
if os.path.exists("paper_embeddings.npy"):
    print("Loading saved embeddings...")

    embeddings = np.load("paper_embeddings.npy")

else:
    print("Generating embeddings...")

    embeddings = model.encode(
        df["paper_text"].tolist(),
        batch_size=32,
        show_progress_bar=True
    )

    np.save("paper_embeddings.npy", embeddings)

    print("Embeddings saved successfully!")

Loading saved embeddings...


In [ ]:
if os.path.exists("paper_faiss.index"):
    print("Loading existing FAISS index...")

    index = faiss.read_index("paper_faiss.index")

else:
    print("Creating new FAISS index...")

    # Normalize embeddings for cosine similarity
    faiss.normalize_L2(embeddings)

    # Create FAISS index
    index = faiss.IndexFlatIP(384)

    # Add embeddings
    index.add(embeddings)

    # Save index
    faiss.write_index(index, "paper_faiss.index")

    print("FAISS index saved successfully!")

Creating new FAISS index...
FAISS index saved successfully!


In [ ]:
print("Total vectors stored:", index.ntotal)

Total vectors stored: 10000


In [ ]:
def search_papers(query, top_k=5):

    # Convert query to embedding
    query_embedding = model.encode([query])

    # Normalize query
    faiss.normalize_L2(query_embedding)

    # Search
    distances, indices = index.search(query_embedding, top_k)

    print("=" * 80)
    print("Query:", query)
    print("=" * 80)

    for i, idx in enumerate(indices[0], 1):

        print(f"\nResult {i}")
        print("-" * 80)

        print("Title:")
        print(df.iloc[idx]["title"])

        print("\nAbstract:")
        print(df.iloc[idx]["abstract"][:500] + "...")

        print("\nSimilarity Score:", round(float(distances[0][i-1]), 4))

In [ ]:
search_papers("Large Language Models")

Query: Large Language Models

Result 1
--------------------------------------------------------------------------------
Title:
LightLDA: Big Topic Models on Modest Compute Clusters

Abstract:
  When building large-scale machine learning (ML) programs, such as big topic
models or deep neural nets, one usually assumes such tasks can only be
attempted with industrial-sized clusters with thousands of nodes, which are out
of reach for most practitioners or academic researchers. We consider this
challenge in the context of topic modeling on web-scale corpora, and show that
with a modest cluster of as few as 8 machines, we can train a topic model with
1 million topics and a 1-million-word v...

Similarity Score: 0.6325

Result 2
--------------------------------------------------------------------------------
Title:
Strategies for Training Large Vocabulary Neural Language Models

Abstract:
  Training neural network language models over large vocabularies is still
computationally very costly co

In [ ]:
search_papers("Deep Learning")

Query: Deep Learning

Result 1
--------------------------------------------------------------------------------
Title:
Deep Learning of Representations: Looking Forward

Abstract:
  Deep learning research aims at discovering learning algorithms that discover
multiple levels of distributed representations, with higher levels representing
more abstract concepts. Although the study of deep learning has already led to
impressive theoretical results, learning algorithms and breakthrough
experiments, several challenges lie ahead. This paper proposes to examine some
of these challenges, centering on the questions of scaling deep learning
algorithms to much larger models and data...

Similarity Score: 0.6159

Result 2
--------------------------------------------------------------------------------
Title:
An exact mapping between the Variational Renormalization Group and Deep
  Learning

Abstract:
  Deep learning is a broad set of techniques that uses multiple layers of
representation to automa

In [ ]:
search_papers("Computer Vision")

Query: Computer Vision

Result 1
--------------------------------------------------------------------------------
Title:
Evolution of active categorical image classification via saccadic eye
  movement

Abstract:
  Pattern recognition and classification is a central concern for modern
information processing systems. In particular, one key challenge to image and
video classification has been that the computational cost of image processing
scales linearly with the number of pixels in the image or video. Here we
present an intelligent machine (the "active categorical classifier," or ACC)
that is inspired by the saccadic movements of the eye, and is capable of
classifying images by selectively scanning only ...

Similarity Score: 0.5171

Result 2
--------------------------------------------------------------------------------
Title:
What you need to know about the state-of-the-art computational models of
  object-vision: A tour through the models

Abstract:
  Models of object vision have b

In [ ]:
search_papers("Neural Networks")

Query: Neural Networks

Result 1
--------------------------------------------------------------------------------
Title:
Neural Networks for Complex Data

Abstract:
  Artificial neural networks are simple and efficient machine learning tools.
Defined originally in the traditional setting of simple vector data, neural
network models have evolved to address more and more difficulties of complex
real world problems, ranging from time evolving data to sophisticated data
structures such as graphs and functions. This paper summarizes advances on
those themes from the last decade, with a focus on results obtained by members
of the SAMM team of Universit\'e Paris 1...

Similarity Score: 0.5668

Result 2
--------------------------------------------------------------------------------
Title:
Highway Networks

Abstract:
  There is plenty of theoretical and empirical evidence that depth of neural
networks is a crucial ingredient for their success. However, network training
becomes more difficult w

In [ ]:
search_papers("Medical AI")

Query: Medical AI

Result 1
--------------------------------------------------------------------------------
Title:
Performance Based Evaluation of Various Machine Learning Classification
  Techniques for Chronic Kidney Disease Diagnosis

Abstract:
  Areas where Artificial Intelligence (AI) & related fields are finding their
applications are increasing day by day, moving from core areas of computer
science they are finding their applications in various other domains.In recent
times Machine Learning i.e. a sub-domain of AI has been widely used in order to
assist medical experts and doctors in the prediction, diagnosis and prognosis
of various diseases and other medical disorders. In this manuscript the authors
applied various machine learni...

Similarity Score: 0.4941

Result 2
--------------------------------------------------------------------------------
Title:
Doctor AI: Predicting Clinical Events via Recurrent Neural Networks

Abstract:
  Leveraging large historical data in electr

In [ ]:
search_papers("Reinforcement Learning")

Query: Reinforcement Learning

Result 1
--------------------------------------------------------------------------------
Title:
Selecting the State-Representation in Reinforcement Learning

Abstract:
  The problem of selecting the right state-representation in a reinforcement
learning problem is considered. Several models (functions mapping past
observations to a finite set) of the observations are given, and it is known
that for at least one of these models the resulting state dynamics are indeed
Markovian. Without knowing neither which of the models is the correct one, nor
what are the probabilistic characteristics of the resulting MDP, it is required
to obtain as much reward as the optima...

Similarity Score: 0.5939

Result 2
--------------------------------------------------------------------------------
Title:
PAC Reinforcement Learning with Rich Observations

Abstract:
  We propose and study a new model for reinforcement learning with rich
observations, generalizing contextual b

In [ ]:
!pip install -q google-genai

In [ ]:
from google import genai

In [2]:


from google.colab import userdata
from google import genai

API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=API_KEY)

print("Gemini Connected Successfully!")

Gemini Connected Successfully!


In [ ]:
paper = df.iloc[0]["paper_text"]

prompt = f"""
You are an AI research assistant.

Analyze the following research paper and provide the response in this format:

1. Paper Title
2. Short Summary (3-4 lines)
3. Main Objective
4. Key Contributions (Bullet Points)
5. Methodology Used
6. Important Findings
7. Limitations
8. Future Scope
9. Explain Like I'm a Beginner

Research Paper:

{paper}
"""

In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

print(response.text)

Here's an analysis of the research paper abstract:

1.  **Paper Title**
    Learning from compressed observations.

2.  **Short Summary**
    This paper addresses statistical learning where the predictor variable $X$ is perfectly observed, but the target variable $Y$ must be communicated at a finite bit rate (i.e., compressed). The core contribution is an information-theoretic framework that characterizes achievable prediction performance in terms of conditional distortion-rate functions. This framework applies under specific regularity conditions and is illustrated using nonparametric regression with Gaussian noise.

3.  **Main Objective**
    To provide an information-theoretic characterization of achievable predictor performance in statistical learning scenarios where the target variable ($Y$) observations are compressed due to finite bit rate constraints, while allowing the encoding of $Y$ to depend on $X$.

4.  **Key Contributions**
    *   Proposes a novel information-theoretic f

In [ ]:
def search_papers(query, top_k=5):

    query_embedding = model.encode([query])

    faiss.normalize_L2(query_embedding)

    distances, indices = index.search(query_embedding, top_k)

    results = []

    print("=" * 80)
    print(f"Query : {query}")
    print("=" * 80)

    for rank, idx in enumerate(indices[0], start=1):

        title = df.iloc[idx]["title"]

        score = float(distances[0][rank-1])

        results.append(idx)

        print(f"\n{rank}. {title}")
        print(f"Similarity Score : {score:.4f}")

    return results

In [ ]:
def summarize_search_result(query, result_number=1):

    results = search_papers(query)

    paper_index = results[result_number-1]

    paper = df.iloc[paper_index]["paper_text"]

    prompt = f"""
You are an AI Research Assistant.

Analyze the following research paper.

Provide:

1. Summary

2. Key Contributions

3. Methodology

4. Important Findings

5. Limitations

6. Future Scope

7. Explain Like I'm a Beginner

Research Paper:

{paper}
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    print("\n")
    print("="*80)
    print("AI SUMMARY")
    print("="*80)

    print(response.text)

In [ ]:
summarize_search_result(
    "Large Language Models",
    1
)

Query : Large Language Models

1. LightLDA: Big Topic Models on Modest Compute Clusters
Similarity Score : 0.6325

2. Strategies for Training Large Vocabulary Neural Language Models
Similarity Score : 0.6223

3. A Fast and Simple Algorithm for Training Neural Probabilistic Language
  Models
Similarity Score : 0.5940

4. Structured Prediction of Sequences and Trees using Infinite Contexts
Similarity Score : 0.5811

5. Model-Parallel Inference for Big Topic Models
Similarity Score : 0.5792


AI SUMMARY
Here's an analysis of the LightLDA research paper:

---

### LightLDA: Big Topic Models on Modest Compute Clusters

---

### 1. Summary

This research paper addresses the significant challenge of building and training large-scale machine learning models, specifically topic models, which traditionally require industrial-sized clusters with thousands of machines. The authors demonstrate a groundbreaking achievement: training a massive topic model with **1 million topics, a 1-million-word voc